# SsisDemo → Fabric Lakehouse migration

Flavor B build pass of the SSIS → Fabric migration. Reads the five source
tables (already exported as Parquet to `Files/raw/` by
`migration/source-to-onelake/export-to-parquet.ps1`), then builds the
three Delta tables under `Tables/`:

* `dim_customer` — SCD2-ready (initial load = full insert).
* `dim_product` — `UPPER(sku)` + derived `margin_category` (replaces the
  C# Script Component from `Load_Products_Scripted.dtsx`).
* `fact_orders` — join against current `dim_customer` for the surrogate
  key.

Expected row counts after a successful run match
`out/baseline/target-rowcounts.json`: 500 / 100 / 2000.

**Attach this notebook to lakehouse `lh_ssis_demo` before running.**

In [ ]:
# Parameters --------------------------------------------------------
# Use workspace + lakehouse IDs to avoid friendly-name resolution issues.
lh_base      = "abfss://c9bd4043-11bf-483d-bfca-1c2b2490c3af@onelake.dfs.fabric.microsoft.com/e3fcb33a-13d8-4967-add9-5efed6fcac65"
raw_dir      = f"{lh_base}/Files/raw"
tables_dir   = f"{lh_base}/Tables"
dim_customer = f"{tables_dir}/dim_customer"
dim_product  = f"{tables_dir}/dim_product"
fact_orders  = f"{tables_dir}/fact_orders"

from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [ ]:
# Read source parquet from the lakehouse Files/raw folder and register as temp views.
for tbl in ["Customers", "Products", "Orders", "OrderItems", "CountryLookup"]:
    df = spark.read.parquet(f"{raw_dir}/{tbl}.parquet")
    df.createOrReplaceTempView(tbl)
    print(f"{tbl}: {df.count()} rows")

In [ ]:
# Build dim_customer (SCD2-ready, initial load = full insert).
spark.sql("DROP TABLE IF EXISTS dim_customer")
src = spark.sql("""
WITH s AS (
    SELECT  C.CustomerID  AS customer_id,
            C.FullName    AS full_name,
            C.Email       AS email,
            CL.CountryName AS country_name,
            current_timestamp() AS valid_from,
            CAST(NULL AS TIMESTAMP) AS valid_to,
            true AS is_current
    FROM Customers C
    LEFT JOIN CountryLookup CL ON CL.CountryName = C.Country
)
SELECT  ROW_NUMBER() OVER (ORDER BY customer_id) AS customer_key,
        customer_id, full_name, email, country_name, valid_from, valid_to, is_current
FROM s
""")
src.write.format("delta").mode("overwrite").option("path", dim_customer).saveAsTable("dim_customer")
print(spark.table("dim_customer").count(), "rows written to dim_customer")

In [ ]:
# Build dim_product. Replaces SSIS C# Script Component with pure SparkSQL.
spark.sql("DROP TABLE IF EXISTS dim_product")
src = spark.sql("""
WITH s AS (
    SELECT  ProductID    AS product_id,
            UPPER(Sku)   AS sku,
            Name         AS name,
            Category     AS category,
            CAST(Price AS DECIMAL(10,2)) AS price,
            CASE WHEN Price <  50 THEN 'LOW' WHEN Price < 200 THEN 'MEDIUM' ELSE 'HIGH' END AS margin_category
    FROM Products
)
SELECT  ROW_NUMBER() OVER (ORDER BY product_id) AS product_key,
        product_id, sku, name, category, price, margin_category
FROM s
""")
src.write.format("delta").mode("overwrite").option("path", dim_product).saveAsTable("dim_product")
print(spark.table("dim_product").count(), "rows written to dim_product")

In [ ]:
# Build fact_orders. LEFT JOIN against current dim_customer slice for surrogate key.
spark.sql("DROP TABLE IF EXISTS fact_orders")
src = spark.sql("""
WITH s AS (
    SELECT  O.OrderID                     AS order_id,
            DC.customer_key               AS customer_key,
            O.OrderDate                   AS order_date,
            CAST(O.TotalAmount AS DECIMAL(12,2)) AS total_amount,
            O.Status                      AS status
    FROM Orders O
    LEFT JOIN (SELECT * FROM dim_customer WHERE is_current = true) DC
      ON DC.customer_id = O.CustomerID
)
SELECT  ROW_NUMBER() OVER (ORDER BY order_id) AS order_key,
        order_id, customer_key, order_date, total_amount, status
FROM s
""")
src.write.format("delta").mode("overwrite").option("path", fact_orders).saveAsTable("fact_orders")
print(spark.table("fact_orders").count(), "rows written to fact_orders")

In [ ]:
# Validation: row counts and orphan check.
spark.sql("""
SELECT 'dim_customer' AS tbl, COUNT(*) AS rows FROM dim_customer
UNION ALL SELECT 'dim_product', COUNT(*) FROM dim_product
UNION ALL SELECT 'fact_orders', COUNT(*) FROM fact_orders
""").show(truncate=False)
spark.sql("SELECT COUNT(*) AS orphan_orders FROM fact_orders WHERE customer_key IS NULL").show()
spark.sql("SELECT margin_category, COUNT(*) AS rows FROM dim_product GROUP BY margin_category").show()

## How to run

1. **Land the source data** by running
   `migration/source-to-onelake/export-to-parquet.ps1` from the dev box.
   That script exports the five `SalesSrc` tables from the VM into
   `out/raw/*.parquet` and uploads them to
   `lh_ssis_demo/Files/raw/`.
2. **Upload this notebook** by running
   `migration/lakehouse/upload-notebook.ps1`. It posts the `.ipynb` to
   workspace `ws-ssis2fabric-demo` via the Fabric REST API.
3. **Open** the notebook in the Fabric portal, **attach** it to lakehouse
   `lh_ssis_demo` (Notebook → Add Lakehouse → select existing), and click
   **Run all**.
4. Compare the validation counts against
   `out/baseline/target-rowcounts.json` (500 / 100 / 2000).